# Talent Intelligence & Skills Gap Analysis
## 01 — Data Quality & Exploration

**Scenario:** NorthStar Digital Services is fictional and all workforce data are synthetic.

### Objectives
1. Load the five source datasets.
2. Validate keys, missing values, ranges, and referential integrity.
3. Confirm every employee and role skill maps to the canonical skills taxonomy.
4. Explore the workforce and skill supply before building the Role Readiness model.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)


### 1. Locate the project data

In [ ]:
# Recommended repository structure:
# project-root/
#   data/raw/
#   notebooks/
#   outputs/

candidates = [
    Path('../data/raw'),  # when notebook is inside /notebooks
    Path('data/raw'),
    Path('..'),           # fallback if CSV files are still in repo root
    Path('.')
]

required_files = {
    'employee_master.csv',
    'employee_skills.csv',
    'skills_taxonomy.csv',
    'target_roles.csv',
    'role_skills.csv'
}

DATA_DIR = None
for candidate in candidates:
    if required_files.issubset({p.name for p in candidate.glob('*.csv')}):
        DATA_DIR = candidate.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError('Could not find all five source CSV files.')

DATA_DIR


### 2. Load datasets

In [ ]:
employee_master = pd.read_csv(DATA_DIR / 'employee_master.csv')
employee_skills = pd.read_csv(DATA_DIR / 'employee_skills.csv')
skills_taxonomy = pd.read_csv(DATA_DIR / 'skills_taxonomy.csv')
target_roles = pd.read_csv(DATA_DIR / 'target_roles.csv')
role_skills = pd.read_csv(DATA_DIR / 'role_skills.csv')

datasets = {
    'employee_master': employee_master,
    'employee_skills': employee_skills,
    'skills_taxonomy': skills_taxonomy,
    'target_roles': target_roles,
    'role_skills': role_skills,
}

inventory = pd.DataFrame([
    {'dataset': name, 'rows': len(df), 'columns': df.shape[1]}
    for name, df in datasets.items()
])
inventory


### 3. Missing-value check

In [ ]:
missing_summary = pd.concat(
    [df.isna().sum().rename(name) for name, df in datasets.items()],
    axis=1
).fillna(0).astype(int)

missing_summary.loc[missing_summary.sum(axis=1) > 0]


### 4. Primary-key and duplicate checks

In [ ]:
duplicate_checks = pd.DataFrame([
    {
        'check': 'Duplicate EmployeeID in employee_master',
        'count': int(employee_master['EmployeeID'].duplicated().sum())
    },
    {
        'check': 'Duplicate EmployeeID + Skill in employee_skills',
        'count': int(employee_skills.duplicated(['EmployeeID', 'Skill']).sum())
    },
    {
        'check': 'Duplicate CanonicalSkill in skills_taxonomy',
        'count': int(skills_taxonomy['CanonicalSkill'].duplicated().sum())
    },
    {
        'check': 'Duplicate RoleID in target_roles',
        'count': int(target_roles['RoleID'].duplicated().sum())
    },
    {
        'check': 'Duplicate RoleID + Skill in role_skills',
        'count': int(role_skills.duplicated(['RoleID', 'Skill']).sum())
    },
])

duplicate_checks


### 5. Referential-integrity checks

In [ ]:
integrity_checks = pd.DataFrame([
    {
        'check': 'Employee-skill records with unknown EmployeeID',
        'count': int((~employee_skills['EmployeeID'].isin(employee_master['EmployeeID'])).sum())
    },
    {
        'check': 'Role-skill records with unknown RoleID',
        'count': int((~role_skills['RoleID'].isin(target_roles['RoleID'])).sum())
    },
    {
        'check': 'Employee skills not in canonical taxonomy',
        'count': len(set(employee_skills['Skill']) - set(skills_taxonomy['CanonicalSkill']))
    },
    {
        'check': 'Role skills not in canonical taxonomy',
        'count': len(set(role_skills['Skill']) - set(skills_taxonomy['CanonicalSkill']))
    },
])

integrity_checks


### 6. Domain and range checks

In [ ]:
range_checks = pd.DataFrame([
    ['Age', employee_master['Age'].min(), employee_master['Age'].max(), 'Expected 18–70'],
    ['TenureYears', employee_master['TenureYears'].min(), employee_master['TenureYears'].max(), 'Expected >= 0'],
    ['PerformanceRating', employee_master['PerformanceRating'].min(), employee_master['PerformanceRating'].max(), 'Expected 1–5'],
    ['ProficiencyLevel', employee_skills['ProficiencyLevel'].min(), employee_skills['ProficiencyLevel'].max(), 'Expected 1–5'],
    ['MonthsSinceLastUse', employee_skills['MonthsSinceLastUse'].min(), employee_skills['MonthsSinceLastUse'].max(), 'Expected >= 0'],
    ['RequiredProficiency', role_skills['RequiredProficiency'].min(), role_skills['RequiredProficiency'].max(), 'Expected 1–5'],
    ['ImportanceWeight', role_skills['ImportanceWeight'].min(), role_skills['ImportanceWeight'].max(), 'Expected 1–3'],
], columns=['field', 'min', 'max', 'rule'])

range_checks


### 7. Workforce profile

In [ ]:
department_profile = (
    employee_master.groupby('Department')
    .agg(
        Headcount=('EmployeeID', 'nunique'),
        AvgTenure=('TenureYears', 'mean'),
        AvgPerformance=('PerformanceRating', 'mean')
    )
    .sort_values('Headcount', ascending=False)
)
department_profile.round(2)


In [ ]:
department_profile['Headcount'].sort_values().plot(kind='barh', figsize=(9, 5))
plt.title('Synthetic Workforce Headcount by Department')
plt.xlabel('Employees')
plt.ylabel('Department')
plt.tight_layout()
plt.show()


### 8. Skill-supply profile

In [ ]:
skill_supply = (
    employee_skills.groupby('Skill')
    .agg(
        EmployeesWithSkill=('EmployeeID', 'nunique'),
        AvgProficiency=('ProficiencyLevel', 'mean'),
        AvgMonthsSinceLastUse=('MonthsSinceLastUse', 'mean')
    )
    .sort_values(['EmployeesWithSkill', 'AvgProficiency'], ascending=False)
)

skill_supply.head(15).round(2)


In [ ]:
skill_supply.head(15)['EmployeesWithSkill'].sort_values().plot(kind='barh', figsize=(9, 6))
plt.title('Top 15 Skills by Employee Coverage')
plt.xlabel('Employees with skill')
plt.ylabel('Skill')
plt.tight_layout()
plt.show()


### 9. Role-demand profile

In [ ]:
role_demand = (
    role_skills.groupby('Skill')
    .agg(
        RolesRequiringSkill=('RoleID', 'nunique'),
        AvgRequiredProficiency=('RequiredProficiency', 'mean'),
        AvgImportanceWeight=('ImportanceWeight', 'mean'),
        MandatoryCount=('Mandatory', lambda s: (s == 'Yes').sum())
    )
    .sort_values(['RolesRequiringSkill', 'MandatoryCount', 'AvgImportanceWeight'], ascending=False)
)

role_demand.head(15).round(2)


### 10. Consolidated QA result
For this synthetic starter dataset, every critical quality check should return **PASS** before we build the matching model.

In [ ]:
qa_results = []

def add_check(name, failures):
    qa_results.append({
        'Check': name,
        'Failures': int(failures),
        'Status': 'PASS' if int(failures) == 0 else 'FAIL'
    })

add_check('EmployeeID is unique in employee_master', employee_master['EmployeeID'].duplicated().sum())
add_check('No duplicate employee-skill pairs', employee_skills.duplicated(['EmployeeID','Skill']).sum())
add_check('Canonical skills are unique', skills_taxonomy['CanonicalSkill'].duplicated().sum())
add_check('RoleID is unique in target_roles', target_roles['RoleID'].duplicated().sum())
add_check('No duplicate role-skill pairs', role_skills.duplicated(['RoleID','Skill']).sum())
add_check('All employee-skill EmployeeIDs exist', (~employee_skills['EmployeeID'].isin(employee_master['EmployeeID'])).sum())
add_check('All role-skill RoleIDs exist', (~role_skills['RoleID'].isin(target_roles['RoleID'])).sum())
add_check('All employee skills map to taxonomy', len(set(employee_skills['Skill']) - set(skills_taxonomy['CanonicalSkill'])))
add_check('All role skills map to taxonomy', len(set(role_skills['Skill']) - set(skills_taxonomy['CanonicalSkill'])))
add_check('No missing values in source tables', sum(int(df.isna().sum().sum()) for df in datasets.values()))
add_check('Employee proficiency is within 1–5', (~employee_skills['ProficiencyLevel'].between(1,5)).sum())
add_check('Required proficiency is within 1–5', (~role_skills['RequiredProficiency'].between(1,5)).sum())
add_check('Importance weight is within 1–3', (~role_skills['ImportanceWeight'].between(1,3)).sum())

qa_results = pd.DataFrame(qa_results)
qa_results


In [ ]:
# Save reusable outputs for the next phase
OUTPUT_DIR = Path('../outputs') if Path('../outputs').parent.exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

qa_results.to_csv(OUTPUT_DIR / 'data_quality_summary.csv', index=False)
department_profile.reset_index().to_csv(OUTPUT_DIR / 'department_profile.csv', index=False)
skill_supply.reset_index().to_csv(OUTPUT_DIR / 'skill_supply_profile.csv', index=False)
role_demand.reset_index().to_csv(OUTPUT_DIR / 'role_demand_profile.csv', index=False)

print('Saved outputs to:', OUTPUT_DIR.resolve())


## Next phase
In Notebook 02 we will build the core analytical engine:

**Employee Skill Profile → Target Role Requirements → Skill Gap → Weighted Role Readiness Score → Internal Candidate Ranking**